In [ ]:
import pandas as pd
import numpy as np
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from scipy.stats import pearsonr


def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]

    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    df = df.dropna(subset=["target"])
    all_moons = np.array(sorted(df["moon"].unique()))

    # ── Config ────────────────────────────────────────────────────────────────
    N_BAGS        = 20     # number of bag models
    MOON_FRAC     = 0.70   # fraction of moons sampled per bag
    WINDOW        = 400    # only use last 400 moons (Step 6 winner)
    RECENCY_STR   = 5.0    # recency weight strength (Step 5 winner)

    # Restrict to last WINDOW moons
    window_moons = all_moons[-WINDOW:].tolist()
    df           = df[df["moon"].isin(window_moons)].copy()
    all_moons    = np.array(sorted(df["moon"].unique()))
    n_moons      = len(all_moons)
    moon_to_rank = {m: i for i, m in enumerate(all_moons)}

    print(f"  Bagging: {N_BAGS} models, {MOON_FRAC:.0%} moon sample each")
    print(f"  Window: last {WINDOW} moons ({all_moons[0]}→{all_moons[-1]})")

    models   = []
    bag_ics  = []

    for bag in range(N_BAGS):
        rng = np.random.default_rng(bag)

        # Sample moons without replacement
        n_sample    = max(50, int(n_moons * MOON_FRAC))
        sample_idx  = rng.choice(n_moons, size=n_sample, replace=False)
        bag_moons   = all_moons[sample_idx].tolist()

        # Held-out = remaining moons (OOB estimate)
        oob_moons = [m for m in all_moons if m not in set(bag_moons)]

        df_bag = df[df["moon"].isin(bag_moons)].copy()
        df_oob = df[df["moon"].isin(oob_moons)].copy()

        # Recency weights
        ranks      = df_bag["moon"].map(moon_to_rank).values
        norm_ranks = ranks / max(ranks.max(), 1)
        weights    = np.exp(norm_ranks * RECENCY_STR)

        m = LGBMRegressor(
            n_estimators=300, learning_rate=0.03, num_leaves=63,
            subsample=0.8, colsample_bytree=0.7,
            random_state=bag, n_jobs=-1, verbose=-1,
        )
        m.fit(df_bag[feature_cols].fillna(0), df_bag["target"],
              sample_weight=weights)

        # OOB IC
        if len(df_oob) > 0:
            corrs = []
            for moon in oob_moons:
                grp = df_oob[df_oob["moon"] == moon]
                if len(grp) < 2: continue
                r, _ = pearsonr(m.predict(grp[feature_cols].fillna(0)), grp["target"])
                if not np.isnan(r): corrs.append(r)
            ic = float(np.mean(corrs)) if corrs else 0.0
        else:
            ic = 0.0
        bag_ics.append(ic)
        models.append(m)
        print(f"    Bag {bag+1:2d}/{N_BAGS}: train={len(bag_moons)} moons  "
              f"OOB={len(oob_moons)} moons  OOB IC={ic:+.5f}")

    print(f"  Mean OOB IC = {np.mean(bag_ics):+.5f}")

    joblib.dump({"models": models, "features": feature_cols},
                os.path.join(model_directory_path, "model.joblib"))
    print("  Saved.")


In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):
    obj      = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    models   = obj["models"]
    features = obj["features"]
    X        = X_test[features].fillna(0)
    preds    = np.mean([m.predict(X) for m in models], axis=0)

    return pd.DataFrame({
        "moon":       X_test["moon"].values,
        "id":         X_test["id"].values,
        "prediction": preds,
    })


In [ ]:
import os
from scipy.stats import pearsonr

embargo   = 4
model_dir = "./model"
os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []
for moon in splits["reduced_cloud"]:
    pred = infer(X_test_cloud[X_test_cloud["moon"] == moon],
                 model_dir, loop_moon=moon, embargo=embargo)
    results.append(pred)

predictions = pd.concat(results)
corrs = []
for moon in splits["reduced_cloud"]:
    merged = predictions[predictions["moon"] == moon].merge(
        y_test_cloud[y_test_cloud["moon"] == moon], on=["moon", "id"])
    if len(merged) > 1:
        r, _ = pearsonr(merged["prediction"], merged["target"])
        corrs.append(r)
        print(f"Moon {moon}: Pearson r = {r:.4f}")
print(f"\nMean IC = {np.mean(corrs):.4f}")
